# Freemarket Medallion Pipeline — orchestrator

Thin orchestrator: all logic lives in the tested `src/` package (see `plan/ARCHITECTURE.md`).
Run top-to-bottom (Restart & Run All) from an empty warehouse.

Bronze `live` = facts + counterparty; Silver `core` = dims + FX match; Silver `shape` =
resolved attributes + GBP-normalised facts; Gold `data_mart` = entity dimension + edge fact; Gold `curated` = network nodes + edges.

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import config, warehouse, bronze, silver_core, silver_shape, gold, gold_curated
from src.pipeline import build_foundation
config.WAREHOUSE_PATH

PosixPath('/Users/hallabehery/Documents/My Projects/FM-Data-Engineer-Test-HS/submission/warehouse.duckdb')

## 1. Foundation — six schemas

In [2]:
con = warehouse.connect()
build_foundation(con)

16:17:33 | INFO  | [foundation] schemas ready: raw, live, core, shape, data_mart, curated


('raw', 'live', 'core', 'shape', 'data_mart', 'curated')

## 2. Bronze `raw` — transactional sheets split per month

In [3]:
for r in bronze.build_raw_monthly(con):
    print(r)

16:17:33 | INFO  | [bronze.raw.deposit] in=6383 out=6383 kept=6383 quarantined=0 dropped=0 conserved=yes


16:17:33 | INFO  | [bronze.raw.withdrawal] in=6599 out=6599 kept=6599 quarantined=0 dropped=0 conserved=yes


[bronze.raw.deposit] in=6383 out=6383 kept=6383 quarantined=0 dropped=0 conserved=yes
[bronze.raw.withdrawal] in=6599 out=6599 kept=6599 quarantined=0 dropped=0 conserved=yes


In [4]:
con.sql("SELECT * FROM raw.deposit_2025_07 LIMIT 10").df()

,freemarket_entity,transaction_type,dc_id,account_id,transaction_id,tx_date,tx_time,tx_currency,tx_value_ccy,counterparty_id,tx_month,scheme
0,UK,Deposit,234939159769,16472.0,343316.0,2025-07-30,13:38:31.593000,USD,2.438954e+06,CP6,2025-07-01,Swift
1,UK,Deposit,234939159769,16472.0,341662.0,2025-07-24,10:39:47.117000,USD,2.736424e+05,CP5,2025-07-01,Swift
2,UK,Deposit,234939159769,16472.0,338562.0,2025-07-14,12:16:16.897000,USD,9.515370e+04,CP5,2025-07-01,Swift
3,UK,Deposit,234939159769,16472.0,336583.0,2025-07-07,12:52:57.710000,USD,1.627457e+06,CP6,2025-07-01,Swift
4,UK,Deposit,234939159769,16472.0,340011.0,2025-07-18,12:50:36.097000,USD,1.209455e+06,CP5,2025-07-01,Swift
5,UK,Deposit,234939159769,16472.0,338879.0,2025-07-15,10:47:03.707000,USD,8.258716e+05,CP6,2025-07-01,Swift
6,UK,Deposit,263002682585,16968.0,339726.0,2025-07-17,15:33:15.537001,EUR,6.032883e+04,CP12,2025-07-01,Sepa
7,UK,Deposit,234850008297,16627.0,340931.0,2025-07-22,09:36:20.080000,USD,4.591582e+04,CP18,2025-07-01,Swift
8,UK,Deposit,263011742938,16914.0,343561.0,2025-07-31,10:18:41.933000,EUR,2.924164e+05,CP39,2025-07-01,SepaInstant
9,UK,Deposit,263011742938,16914.0,337205.0,2025-07-09,09:09:09.650000,EUR,3.899708e+05,CP39,2025-07-01,SepaInstant


## 3. Bronze `live` — consolidate + land counterparty & fee (facts live here)

In [5]:
for r in bronze.build_live(con):
    print(r)

16:17:33 | INFO  | [bronze.live.deposit] in=6383 out=6383 kept=6383 quarantined=0 dropped=0 conserved=yes


16:17:33 | INFO  | [bronze.live.withdrawal] in=6599 out=6599 kept=6599 quarantined=0 dropped=0 conserved=yes


16:17:33 | INFO  | [bronze.live.counterparty] in=1585 out=1585 kept=1585 quarantined=0 dropped=0 conserved=yes


16:17:33 | INFO  | [bronze.live.fee] in=21921 out=21921 kept=21921 quarantined=0 dropped=0 conserved=yes


[bronze.live.deposit] in=6383 out=6383 kept=6383 quarantined=0 dropped=0 conserved=yes
[bronze.live.withdrawal] in=6599 out=6599 kept=6599 quarantined=0 dropped=0 conserved=yes
[bronze.live.counterparty] in=1585 out=1585 kept=1585 quarantined=0 dropped=0 conserved=yes
[bronze.live.fee] in=21921 out=21921 kept=21921 quarantined=0 dropped=0 conserved=yes


In [6]:
con.sql("SELECT * FROM live.deposit LIMIT 10").df()

,freemarket_entity,transaction_type,dc_id,account_id,transaction_id,tx_date,tx_time,tx_currency,tx_value_ccy,counterparty_id,tx_month,scheme
0,UK,Deposit,234939159769,16472.0,343316.0,2025-07-30,13:38:31.593000,USD,2.438954e+06,CP6,2025-07-01,Swift
1,UK,Deposit,234939159769,16472.0,341662.0,2025-07-24,10:39:47.117000,USD,2.736424e+05,CP5,2025-07-01,Swift
2,UK,Deposit,234939159769,16472.0,338562.0,2025-07-14,12:16:16.897000,USD,9.515370e+04,CP5,2025-07-01,Swift
3,UK,Deposit,234939159769,16472.0,336583.0,2025-07-07,12:52:57.710000,USD,1.627457e+06,CP6,2025-07-01,Swift
4,UK,Deposit,234939159769,16472.0,340011.0,2025-07-18,12:50:36.097000,USD,1.209455e+06,CP5,2025-07-01,Swift
5,UK,Deposit,234939159769,16472.0,338879.0,2025-07-15,10:47:03.707000,USD,8.258716e+05,CP6,2025-07-01,Swift
6,UK,Deposit,263002682585,16968.0,339726.0,2025-07-17,15:33:15.537001,EUR,6.032883e+04,CP12,2025-07-01,Sepa
7,UK,Deposit,234850008297,16627.0,340931.0,2025-07-22,09:36:20.080000,USD,4.591582e+04,CP18,2025-07-01,Swift
8,UK,Deposit,263011742938,16914.0,343561.0,2025-07-31,10:18:41.933000,EUR,2.924164e+05,CP39,2025-07-01,SepaInstant
9,UK,Deposit,263011742938,16914.0,337205.0,2025-07-09,09:09:09.650000,EUR,3.899708e+05,CP39,2025-07-01,SepaInstant


## 4. Silver `core` — company & corporate_group unpick (dimensions)

In [7]:
print(silver_core.build_companies(con))
print(silver_core.build_groups(con))

16:17:33 | INFO  | [silver.core.company] in=44 out=44 kept=44 quarantined=0 dropped=0 conserved=yes


16:17:33 | INFO  | [silver.core.corporate_group] in=13 out=13 kept=13 quarantined=0 dropped=0 conserved=yes


[silver.core.company] in=44 out=44 kept=44 quarantined=0 dropped=0 conserved=yes
[silver.core.corporate_group] in=13 out=13 kept=13 quarantined=0 dropped=0 conserved=yes


## 5. Silver `core` — `exchange_rate` dimension

In [8]:
print(silver_core.build_exchange_rates(con))

16:17:34 | INFO  | [silver.core.exchange_rate] in=161342 out=161342 kept=161342 quarantined=0 dropped=0 conserved=yes


[silver.core.exchange_rate] in=161342 out=161342 kept=161342 quarantined=0 dropped=0 conserved=yes


## 6. Silver `core` — FX as-of match (facts stay in `live`)

In [9]:
for r in silver_core.build_fx_match(con):
    print(r)

16:17:35 | INFO  | [silver.core.deposit_fx] in=6383 out=6383 kept=6383 quarantined=0 dropped=0 conserved=yes


16:17:35 | INFO  | [silver.core.withdrawal_fx] in=6599 out=6599 kept=6599 quarantined=0 dropped=0 conserved=yes


16:17:35 | INFO  | [silver.core.fee_fx] in=21921 out=21921 kept=21921 quarantined=0 dropped=0 conserved=yes


16:17:35 | INFO  | [silver.core.fee_fx] flagged for FX quarantine (kept, not dropped): 42


[silver.core.deposit_fx] in=6383 out=6383 kept=6383 quarantined=0 dropped=0 conserved=yes
[silver.core.withdrawal_fx] in=6599 out=6599 kept=6599 quarantined=0 dropped=0 conserved=yes
[silver.core.fee_fx] in=21921 out=21921 kept=21921 quarantined=0 dropped=0 conserved=yes


In [10]:
con.sql("SELECT * FROM core.deposit_fx LIMIT 10").df()

,transaction_id,fx_instant_ms,fx_rate_id,fx_rate,fx_quarantine_reason
0,343316.0,1753882711593,13848698,0.752064,<NA>
1,341662.0,1753353587117,13832598,0.738758,<NA>
2,338562.0,1752495376897,13801564,0.741411,<NA>
3,336583.0,1751892777710,13781902,0.734611,<NA>
4,340011.0,1752843036097,13817325,0.743028,<NA>
5,338879.0,1752576423707,13805164,0.743192,<NA>
6,339726.0,1752766395537,13813856,0.864524,<NA>
7,340931.0,1753176980080,13824631,0.741222,<NA>
8,343561.0,1753957121933,13851592,0.864840,<NA>
9,337205.0,1752052149650,13788912,0.861393,<NA>


## 7. Silver `shape` — resolve heterogeneous entity attributes

In [11]:
for r in silver_shape.build_entity_shape(con):
    print(r)

16:17:35 | INFO  | [silver.shape.company] in=44 out=44 kept=44 quarantined=0 dropped=0 conserved=yes


16:17:35 | INFO  | [silver.shape.corporate_group] in=13 out=13 kept=13 quarantined=0 dropped=0 conserved=yes


[silver.shape.company] in=44 out=44 kept=44 quarantined=0 dropped=0 conserved=yes
[silver.shape.corporate_group] in=13 out=13 kept=13 quarantined=0 dropped=0 conserved=yes


## 8. Silver `shape` — FX applied → GBP normalisation

In [12]:
for r in silver_shape.build_gbp_facts(con):
    print(r)

16:17:35 | INFO  | [silver.shape.deposit] in=6383 out=6383 kept=6383 quarantined=0 dropped=0 conserved=yes


16:17:35 | INFO  | [silver.shape.withdrawal] in=6599 out=6599 kept=6599 quarantined=0 dropped=0 conserved=yes


16:17:35 | INFO  | [silver.shape.fee] in=21921 out=21879 kept=21879 quarantined=42 dropped=0 conserved=yes


[silver.shape.deposit] in=6383 out=6383 kept=6383 quarantined=0 dropped=0 conserved=yes
[silver.shape.withdrawal] in=6599 out=6599 kept=6599 quarantined=0 dropped=0 conserved=yes
[silver.shape.fee] in=21921 out=21879 kept=21879 quarantined=42 dropped=0 conserved=yes


In [13]:
con.sql("SELECT * FROM shape.deposit LIMIT 10").df()

,freemarket_entity,transaction_type,dc_id,account_id,transaction_id,tx_date,tx_time,tx_currency,tx_value_ccy,counterparty_id,tx_month,scheme,fx_rate_id,fx_rate,gbp_amount
0,UK,Deposit,234939159769,16472.0,343316.0,2025-07-30,13:38:31.593000,USD,2.438954e+06,CP6,2025-07-01,Swift,13848698,0.752064,1.834249e+06
1,UK,Deposit,234939159769,16472.0,341662.0,2025-07-24,10:39:47.117000,USD,2.736424e+05,CP5,2025-07-01,Swift,13832598,0.738758,2.021556e+05
2,UK,Deposit,234939159769,16472.0,338562.0,2025-07-14,12:16:16.897000,USD,9.515370e+04,CP5,2025-07-01,Swift,13801564,0.741411,7.054805e+04
3,UK,Deposit,234939159769,16472.0,336583.0,2025-07-07,12:52:57.710000,USD,1.627457e+06,CP6,2025-07-01,Swift,13781902,0.734611,1.195547e+06
4,UK,Deposit,234939159769,16472.0,340011.0,2025-07-18,12:50:36.097000,USD,1.209455e+06,CP5,2025-07-01,Swift,13817325,0.743028,8.986596e+05
5,UK,Deposit,234939159769,16472.0,338879.0,2025-07-15,10:47:03.707000,USD,8.258716e+05,CP6,2025-07-01,Swift,13805164,0.743192,6.137808e+05
6,UK,Deposit,263002682585,16968.0,339726.0,2025-07-17,15:33:15.537001,EUR,6.032883e+04,CP12,2025-07-01,Sepa,13813856,0.864524,5.215570e+04
7,UK,Deposit,234850008297,16627.0,340931.0,2025-07-22,09:36:20.080000,USD,4.591582e+04,CP18,2025-07-01,Swift,13824631,0.741222,3.403382e+04
8,UK,Deposit,263011742938,16914.0,343561.0,2025-07-31,10:18:41.933000,EUR,2.924164e+05,CP39,2025-07-01,SepaInstant,13851592,0.864840,2.528935e+05
9,UK,Deposit,263011742938,16914.0,337205.0,2025-07-09,09:09:09.650000,EUR,3.899708e+05,CP39,2025-07-01,SepaInstant,13788912,0.861393,3.359181e+05


## 9. Gold `data_mart` — combined entity/node dimension (groups + companies + counterparties)

In [14]:
print(gold.build_entity(con))

16:17:35 | INFO  | [gold.data_mart.entity] in=1642 out=1642 kept=1642 quarantined=0 dropped=0 conserved=yes


[gold.data_mart.entity] in=1642 out=1642 kept=1642 quarantined=0 dropped=0 conserved=yes


In [15]:
con.sql('''SELECT entity_kind, node_shape, is_standalone, COUNT(*) n
           FROM data_mart.entity GROUP BY 1,2,3 ORDER BY 1,2''').df()

,entity_kind,node_shape,is_standalone,n
0,company,NaN,False,44
1,counterparty,diamond,True,1363
2,counterparty,NaN,False,222
3,group,circle,False,13


## 10. Gold `data_mart` — directed edge fact
Grain: `focal_group × counterpart × direction × month`. Measures: `gbp_volume`, `txn_count`,
`gbp_fee_revenue` (additive; year rolls up by summation). Direction: deposit = **inflow**,
withdrawal = **outflow**. Counterparts resolve to a group (circle) or stand alone (diamond).

In [16]:
print(gold.build_edge_fact(con))

16:17:35 | INFO  | [gold.data_mart.edge_fact] 12982 transactions -> 4110 edges; gbp_volume / txn_count / gbp_fee_revenue reconciled to Silver


16:17:35 | INFO  | [gold.data_mart.edge_fact] in=4110 out=4110 kept=4110 quarantined=0 dropped=0 conserved=yes


[gold.data_mart.edge_fact] in=4110 out=4110 kept=4110 quarantined=0 dropped=0 conserved=yes


In [17]:
con.sql("SELECT table_name FROM duckdb_tables() WHERE schema_name = 'data_mart' ORDER BY table_name").df()

,table_name
0,edge_fact
1,entity


In [18]:
# Top edges by GBP volume (the star-map figures: £ volume, count, fee)
con.sql('''
    SELECT focal_group_id, counterpart_id, counterpart_is_group, direction,
           round(gbp_volume, 2) AS gbp_volume, txn_count,
           round(gbp_fee_revenue, 2) AS gbp_fee_revenue, source
    FROM data_mart.edge_fact ORDER BY gbp_volume DESC LIMIT 10
''').df()

,focal_group_id,counterpart_id,counterpart_is_group,direction,gbp_volume,txn_count,gbp_fee_revenue,source
0,167898894567,CP465,False,outflow,1.507733e+10,107,146154.36,"[fees, groups.json, companies.json, counterpar..."
1,208399264973,CP522,False,outflow,1.985420e+09,44,40387.32,"[fees, groups.json, companies.json, counterpar..."
2,167898894567,CP5,False,inflow,1.001497e+09,46,39631.27,"[fees, groups.json, companies.json, counterpar..."
3,192407450811,CP39,False,inflow,4.377203e+08,233,14009.88,"[fees, groups.json, companies.json, counterpar..."
4,167898894567,CP6,False,inflow,4.043000e+08,38,19471.64,"[fees, groups.json, companies.json, counterpar..."
5,192400133342,192212183238,True,inflow,3.480963e+08,17,9069.47,"[fees, groups.json, companies.json, deposits]"
6,192402023626,CP60,False,inflow,2.990040e+08,19,4962.96,"[fees, groups.json, companies.json, counterpar..."
7,187752508616,CP1209,False,outflow,2.574102e+08,5,3236.23,"[fees, groups.json, companies.json, counterpar..."
8,192407450811,CP38,False,inflow,2.403337e+08,510,28960.31,"[fees, groups.json, companies.json, counterpar..."
9,167898894567,CP5,False,inflow,2.100665e+08,62,46587.10,"[fees, groups.json, companies.json, counterpar..."


### Slice by year (roll up months by summation) — one focal group

In [19]:
con.sql('''
    SELECT focal_group_id, counterpart_id, direction,
           round(SUM(gbp_volume), 2) AS gbp_volume_year, SUM(txn_count) AS txn_count_year
    FROM data_mart.edge_fact
    WHERE focal_group_id = (SELECT focal_group_id FROM data_mart.edge_fact
                            GROUP BY 1 ORDER BY SUM(gbp_volume) DESC LIMIT 1)
    GROUP BY 1,2,3 ORDER BY gbp_volume_year DESC LIMIT 8
''').df()

,focal_group_id,counterpart_id,direction,gbp_volume_year,txn_count_year
0,167898894567,CP465,outflow,1.577485e+10,560.0
1,167898894567,CP5,inflow,1.746943e+09,218.0
2,167898894567,CP6,inflow,7.243034e+08,128.0
3,167898894567,CP10,inflow,3.096439e+08,62.0
4,167898894567,CP250,inflow,2.126210e+08,25.0
5,167898894567,CP1011,outflow,1.884149e+08,259.0
6,167898894567,CP21,inflow,1.530531e+08,95.0
7,167898894567,CP35,inflow,1.374780e+08,78.0


## 11. Gold `curated` — network nodes
The presentation layer reads **only** from `data_mart`. `curated.node` is one row per
node that participates in an edge: **circles** for our groups, **diamonds** for
standalone counterparties. Companies and group-resolved counterparties are not nodes.

In [20]:
print(gold_curated.build_node(con))

16:17:35 | INFO  | [gold.curated.node] in=1376 out=1376 kept=1376 quarantined=0 dropped=0 conserved=yes


[gold.curated.node] in=1376 out=1376 kept=1376 quarantined=0 dropped=0 conserved=yes


In [21]:
con.sql('''SELECT node_shape, node_kind, is_standalone, COUNT(*) n
           FROM curated.node GROUP BY 1,2,3 ORDER BY 1''').df()

,node_shape,node_kind,is_standalone,n
0,circle,group,False,13
1,diamond,counterparty,True,1363


## 12. Gold `curated` — directed edges (slicing & drill)
The edge side of the final product, read **only** from `data_mart.edge_fact`. Each edge is
directed (`source_node_id → target_node_id`), carries GBP volume / count / fee revenue, is
sliceable by `month` and `year`, and is drillable via `focal_group_id` (roll up) ↔
`focal_company_id` (the entity that actually transacted).

In [22]:
print(gold_curated.build_edge(con))

16:17:35 | INFO  | [gold.curated.edge] in=4110 out=4110 kept=4110 quarantined=0 dropped=0 conserved=yes


[gold.curated.edge] in=4110 out=4110 kept=4110 quarantined=0 dropped=0 conserved=yes


In [23]:
# Directed group-level view for the top focal group (the star-map shape: £ volume, count).
con.sql('''
    SELECT source_node_id, target_node_id, direction, counterpart_is_group,
           round(SUM(gbp_volume), 2) AS gbp_volume, SUM(txn_count) AS txn_count,
           round(SUM(gbp_fee_revenue), 2) AS gbp_fee_revenue
    FROM curated.edge
    WHERE focal_group_id = (SELECT focal_group_id FROM curated.edge
                            GROUP BY 1 ORDER BY SUM(gbp_volume) DESC LIMIT 1)
    GROUP BY 1,2,3,4 ORDER BY gbp_volume DESC LIMIT 10
''').df()

,source_node_id,target_node_id,direction,counterpart_is_group,gbp_volume,txn_count,gbp_fee_revenue
0,167898894567,CP465,outflow,False,1.577485e+10,560.0,151990.60
1,CP5,167898894567,inflow,False,1.746943e+09,218.0,223263.28
2,CP6,167898894567,inflow,False,7.243034e+08,128.0,84122.53
3,CP10,167898894567,inflow,False,3.096439e+08,62.0,42066.39
4,CP250,167898894567,inflow,False,2.126210e+08,25.0,6267.45
5,167898894567,CP1011,outflow,False,1.884149e+08,259.0,2294.33
6,CP21,167898894567,inflow,False,1.530531e+08,95.0,54531.54
7,CP35,167898894567,inflow,False,1.374780e+08,78.0,43211.50
8,CP32,167898894567,inflow,False,1.001724e+08,76.0,28101.40
9,CP269,167898894567,inflow,False,1.000265e+08,19.0,2481.97


### Drill down: same focal group, resolved to the direct companies that transacted

In [24]:
con.sql('''
    SELECT focal_company_id, counterpart_id, direction,
           round(SUM(gbp_volume), 2) AS gbp_volume, SUM(txn_count) AS txn_count
    FROM curated.edge
    WHERE focal_group_id = (SELECT focal_group_id FROM curated.edge
                            GROUP BY 1 ORDER BY SUM(gbp_volume) DESC LIMIT 1)
    GROUP BY 1,2,3 ORDER BY gbp_volume DESC LIMIT 10
''').df()

,focal_company_id,counterpart_id,direction,gbp_volume,txn_count
0,234850008297,CP465,outflow,1.577485e+10,560.0
1,234939159769,CP5,inflow,1.575649e+09,189.0
2,234939159769,CP6,inflow,7.243034e+08,128.0
3,234939159769,CP10,inflow,3.096439e+08,62.0
4,234850008297,CP250,inflow,2.126210e+08,25.0
5,234850008297,CP1011,outflow,1.884149e+08,259.0
6,234850008297,CP5,inflow,1.712932e+08,29.0
7,234850008297,CP21,inflow,1.530531e+08,95.0
8,234850008297,CP35,inflow,1.374780e+08,78.0
9,234850008297,CP32,inflow,1.001724e+08,76.0


## Inspect the built warehouse — table counts per schema

In [25]:
con.close()
import duckdb
inspect = duckdb.connect(str(config.WAREHOUSE_PATH), read_only=True)
tables = inspect.sql('''SELECT table_schema, COUNT(*) AS n_tables
    FROM information_schema.tables GROUP BY table_schema ORDER BY table_schema''').df()
inspect.close()
tables

,table_schema,n_tables
0,core,6
1,curated,2
2,data_mart,2
3,live,4
4,raw,12
5,shape,8
